# Does a metaheuristic feature search beat the frozen combination?

This notebook answers that with a measurement rather than an argument, and the
answer it is built to expose is **no**: a genetic algorithm raises the score it
optimises by a large margin while leaving held-out performance unchanged.

## Why the question is worth settling

The proposed arm of phase 1 searches 15 feature-group combinations
exhaustively, so no search can improve on it. The tempting target for a
metaheuristic is the **column** space underneath: 20 AAC + 400 DPC + 4
propensity + 42 flank + peptide length = **467 columns**, hence 2^467 subsets.
Exhaustive enumeration is impossible there, which is exactly the situation
metaheuristics exist for.

The reason to be sceptical is the signal-to-noise ratio on this data. The
standard error of a five-fold mean ROC-AUC here is about **0.013**, while the
whole spread between the best and worst of the 15 pre-registered combinations
is about **0.013-0.015**. The quantity a search would maximise is the same size
as the uncertainty in each measurement of it, so a search has little real
signal to find and a great deal of noise to exploit.

## Three experiments

**A — the naive use.** Run the GA on all development rows. Its objective is
ROC-AUC on one protein-grouped holdout, which is the cheap fitness a search of
this size can afford, and the result is reported two ways: on that objective,
and as the full grouped five-fold CV of the subset it returns. PROP + LEN is
measured identically. Both are optimistic, the objective most of all, because
the subset was chosen on the very rows it is then scored on. A GA whose
objective were the five-fold mean would inflate this further, so what A shows
is a floor, not a ceiling.

**B — the honest test.** Protein-grouped outer folds. Inside each outer
*training* set the GA searches again, from scratch, and the subset it returns
is scored once on the held-out fold it never saw. PROP + LEN is scored through
the identical loop. If the search finds real signal, B must favour it.

**C — the null control.** The same GA on **shuffled labels**, where by
construction no subset is better than any other. Whatever it reaches is pure
selection inflation, and it puts a number on how much of A is an artefact.

## How to read the output

A reports an optimistic number, B reports an honest one, C reports the size of
the gap between them that is noise. The claim is demonstrated if A shows a
large gain, B shows none, and C explains A.

## Cost

`POPULATION` and `GENERATIONS` set the budget. The cell times one fitness
evaluation and prints an estimate before the long loops start, so the budget
can be cut if the estimate is too large for the session. Defaults are chosen to
finish inside a normal CPU session.

## On the environment

A mismatched Kaggle image warns here instead of stopping. Both sides of every
comparison are measured in whichever environment this runs in, so a version
difference moves them together and the contrast is unaffected. Only the
absolute values would shift, and none of them are reported as study results.

This notebook writes nothing into `ML/artifacts` and changes no module. It
reads B-cell only; SARS is never loaded.


In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or "
            "point PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from ML.CFG import CFG
from ML.dataset import load_labeled_dataset, make_cross_validation_splits
from ML.experiment_matrix import report_environment
from ML.model import build_model
from ML.pre_dataset import build_feature_frame
from ML.selection import resolve_scale_pos_weight

# ---- 2. Settings ------------------------------------------------------------
# The search budget. Every number below is a knob, not a result: the point of
# the notebook is that a LARGER budget makes experiment A look better and
# leaves experiment B where it is.
POPULATION = 20
GENERATIONS = 15
MUTATION_RATE = 0.01          # per-bit; 467 bits, so ~5 flips per child
TOURNAMENT = 3
INITIAL_DENSITY = 0.10        # fraction of columns switched on in generation 0
OUTER_SPLITS = 5
SEARCH_MODEL = "xgboost"      # one model everywhere, so nothing is confounded
RANDOM_SEED = 42

RUN_A_NAIVE = True
RUN_B_NESTED = True
RUN_C_NULL = True

cfg = CFG(MODEL_TYPE=SEARCH_MODEL, DATASET_NAME="bcell")
cfg.validate()

print("Environment:")
if not report_environment():
    print()
    print("NOT the pinned image. That is tolerated here: every comparison below")
    print("is measured within this same environment, so a version difference")
    print("moves both sides together and the contrast is unaffected.")
print()


def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one {filename}; found {len(found)}.")
    return found[0].parent


DATA_DIR = _find_data_dir("input_bcell.csv")

# ---- 3. The data and the searchable column space ----------------------------
frame = load_labeled_dataset(
    "bcell", DATA_DIR, max_peptide_length=cfg.MAX_PEPTIDE_LENGTH,
    feature_set="proposed",
)
features = build_feature_frame(
    frame, feature_set="proposed",
    use_aac_features=True, use_dpc_features=True, use_prop_features=True,
    use_flank_features=True, use_len_features=True,
    use_peptide_features=False, use_protein_features=False,
    use_biological_features=False,
)
labels = frame["target"].to_numpy(dtype=int)
groups = frame["parent_protein_id"].to_numpy()
columns = list(features.columns)
n_columns = len(columns)

# The frozen combination, expressed as a mask over the same column space, so
# the search and the baseline are compared on identical machinery.
FROZEN_COLUMNS = ["chou_fasman", "emini", "kolaskar_tongaonkar", "parker",
                  "peptide_length"]
missing = [c for c in FROZEN_COLUMNS if c not in columns]
if missing:
    raise RuntimeError(f"Frozen columns absent from the space: {missing}")
frozen_mask = np.array([c in set(FROZEN_COLUMNS) for c in columns])

print(f"rows {len(frame):,}   proteins {frame.parent_protein_id.nunique():,}   "
      f"classes {dict(pd.Series(labels).value_counts().sort_index())}")
print(f"searchable columns {n_columns}  ->  2^{n_columns} subsets")
print(f"frozen baseline    {frozen_mask.sum()} columns "
      f"({', '.join(FROZEN_COLUMNS)})")
print()


# ---- 4. Scoring helpers -----------------------------------------------------
def _fit_predict(train_index, test_index, mask, y):
    """One fit of the configured model on the masked columns."""
    chosen = [columns[i] for i in np.flatnonzero(mask)]
    x_train = features.iloc[train_index][chosen]
    x_test = features.iloc[test_index][chosen]
    model = build_model(
        model_type=SEARCH_MODEL,
        random_state=cfg.RANDOM_STATE,
        class_weight=cfg.CLASS_WEIGHT,
        xgboost_estimators=cfg.XGBOOST_ESTIMATORS,
        xgboost_max_depth=cfg.XGBOOST_MAX_DEPTH,
        xgboost_learning_rate=cfg.XGBOOST_LEARNING_RATE,
        xgboost_subsample=cfg.XGBOOST_SUBSAMPLE,
        xgboost_colsample_bytree=cfg.XGBOOST_COLSAMPLE_BYTREE,
        xgboost_scale_pos_weight=resolve_scale_pos_weight(cfg, y[train_index]),
        feature_set="proposed",
    )
    model.fit(x_train, y[train_index])
    return model.predict_proba(x_test)[:, 1]


def grouped_folds(index, y, n_splits, seed):
    """Protein-grouped folds over a subset of rows, as the protocol uses."""
    folds, _ = make_cross_validation_splits(
        y[index], groups[index], "bcell", n_splits=n_splits, random_state=seed,
    )
    return [(index[a], index[b]) for a, b in folds]


def cv_fold_scores(mask, index, y, n_splits, seed):
    """ROC-AUC on each protein-grouped fold, kept per fold, not averaged.

    A fold whose test side carries one class only cannot be scored, so it
    becomes nan and drops out of the mean instead of counting as a zero.
    """
    if mask.sum() == 0:
        return np.full(n_splits, np.nan)
    scores = []
    for train_index, test_index in grouped_folds(index, y, n_splits, seed):
        if len(np.unique(y[test_index])) < 2:
            scores.append(np.nan)
            continue
        scores.append(roc_auc_score(y[test_index],
                                    _fit_predict(train_index, test_index, mask, y)))
    return np.array(scores, dtype=float)


def cv_score(mask, index, y, n_splits, seed):
    """Mean ROC-AUC over protein-grouped folds of the given rows."""
    values = cv_fold_scores(mask, index, y, n_splits, seed)
    return float(np.nanmean(values)) if np.any(~np.isnan(values)) else 0.0


def fold_summary(values):
    """mean, std and SE across folds, defined as phase 1 defines them.

    std is the population std (np.std, ddof=0) to match ML/train.py, and
    SE = std / sqrt(folds) to match ML/select_configuration.py.
    """
    values = np.asarray(values, dtype=float)
    count = int(np.count_nonzero(~np.isnan(values)))
    if not count:
        return float("nan"), float("nan"), float("nan")
    std = float(np.nanstd(values))
    return float(np.nanmean(values)), std, std / np.sqrt(count)


def fold_row(name, values):
    """One row of a per-fold table: each fold, then mean, std and SE."""
    mean, std, se = fold_summary(values)
    row = {"subset": name}
    row.update({f"fold{i}": v for i, v in enumerate(values, start=1)})
    row.update({"mean": mean, "std": std, "se": se})
    return row


def holdout_score(mask, index, y, seed):
    """One grouped split of `index`: the cheap objective the search optimises."""
    folds = grouped_folds(index, y, 5, seed)
    train_index, test_index = folds[0]
    if len(np.unique(y[test_index])) < 2:
        return 0.0
    return float(roc_auc_score(y[test_index],
                               _fit_predict(train_index, test_index, mask, y)))


# ---- 5. The genetic algorithm -----------------------------------------------
def genetic_search(index, y, seed, label, budget_note=""):
    """A plain GA over column masks, maximising a grouped holdout ROC-AUC.

    Deliberately ordinary: tournament selection, uniform crossover, bit-flip
    mutation, elitism. The claim under test is about what ANY search does to a
    noisy objective, not about this particular operator set, and a fancier
    search would only take a larger maximum over the same noise.
    """
    rng = np.random.default_rng(seed)
    population = rng.random((POPULATION, n_columns)) < INITIAL_DENSITY
    # Seed one individual with the frozen combination so the search starts with
    # the incumbent in hand and can only report an improvement on it.
    population[0] = frozen_mask.copy()
    for row in population:
        if not row.any():
            row[rng.integers(n_columns)] = True

    fitness = np.array([holdout_score(m, index, y, seed) for m in population])
    history = [float(fitness.max())]
    evaluations = len(population)

    for generation in range(GENERATIONS):
        children = []
        while len(children) < POPULATION - 1:
            picks = rng.integers(0, POPULATION, size=(2, TOURNAMENT))
            parents = [population[p[np.argmax(fitness[p])]] for p in picks]
            cut = rng.random(n_columns) < 0.5
            child = np.where(cut, parents[0], parents[1])
            flip = rng.random(n_columns) < MUTATION_RATE
            child = np.logical_xor(child, flip)
            if not child.any():
                child[rng.integers(n_columns)] = True
            children.append(child)
        children = np.array(children)
        child_fitness = np.array([holdout_score(m, index, y, seed) for m in children])
        evaluations += len(children)

        elite = int(np.argmax(fitness))
        population = np.vstack([population[elite], children])
        fitness = np.concatenate([[fitness[elite]], child_fitness])
        history.append(float(fitness.max()))
        print(f"    {label} gen {generation + 1:>2}/{GENERATIONS}  "
              f"best={fitness.max():.4f}  features={int(population[int(np.argmax(fitness))].sum())}"
              f"{budget_note}")

    best = population[int(np.argmax(fitness))]
    return best, float(fitness.max()), history, evaluations


# ---- 6. Runtime estimate before committing ----------------------------------
all_index = np.arange(len(frame))
started = time.time()
_ = holdout_score(frozen_mask, all_index, labels, RANDOM_SEED)
cheap = time.time() - started
started = time.time()
_ = holdout_score(np.ones(n_columns, dtype=bool), all_index, labels, RANDOM_SEED)
dear = time.time() - started
per_eval = (cheap + dear) / 2

ga_evals = POPULATION * (GENERATIONS + 1)
planned = (RUN_A_NAIVE * ga_evals
           + RUN_B_NESTED * OUTER_SPLITS * ga_evals
           + RUN_C_NULL * ga_evals)
print(f"one evaluation: {cheap:.1f}s at {int(frozen_mask.sum())} columns, "
      f"{dear:.1f}s at {n_columns}")
print(f"planned evaluations: {planned:,}  ->  rough estimate "
      f"{planned * per_eval / 60:.0f} min (plus the scoring passes)")
print("Reduce POPULATION or GENERATIONS now if that does not fit the session.")
print()

results = {}

# ---- 7. Experiment A: the naive use -----------------------------------------
if RUN_A_NAIVE:
    print("=" * 78)
    print("A. GA on all development rows, reported without nesting")
    print("=" * 78)
    best_a, obj_a, hist_a, _ = genetic_search(all_index, labels, RANDOM_SEED, "A")
    frozen_obj = holdout_score(frozen_mask, all_index, labels, RANDOM_SEED)
    ga_folds = cv_fold_scores(best_a, all_index, labels, OUTER_SPLITS, RANDOM_SEED)
    frozen_folds = cv_fold_scores(
        frozen_mask, all_index, labels, OUTER_SPLITS, RANDOM_SEED)
    ga_cv, ga_cv_sd, ga_cv_se = fold_summary(ga_folds)
    frozen_cv, frozen_cv_sd, frozen_cv_se = fold_summary(frozen_folds)
    results["A"] = {
        "ga_features": int(best_a.sum()),
        "ga_objective": obj_a,
        "frozen_objective": frozen_obj,
        "objective_gain": obj_a - frozen_obj,
        "ga_cv": ga_cv,
        "ga_cv_std": ga_cv_sd,
        "frozen_cv": frozen_cv,
        "frozen_cv_std": frozen_cv_sd,
        "frozen_cv_se": frozen_cv_se,
        "apparent_gain": ga_cv - frozen_cv,
    }
    print()
    print("  on the objective the GA maximised (one grouped holdout):")
    print(f"    GA subset    {int(best_a.sum()):>4} columns   {obj_a:.4f}")
    print(f"    PROP + LEN   {int(frozen_mask.sum()):>4} columns   {frozen_obj:.4f}")
    print(f"    gain                            {obj_a - frozen_obj:+.4f}"
          "   <- what a naive report quotes")
    print("  the same subset on the full grouped 5-fold CV, per fold and then")
    print("  mean +/- std as phase 1 reports it:")
    a_table = pd.DataFrame([
        fold_row(f"GA ({int(best_a.sum())} col)", ga_folds),
        fold_row(f"PROP + LEN ({int(frozen_mask.sum())} col)", frozen_folds),
    ]).set_index("subset")
    print(a_table.round(4).to_string())
    print(f"    GA subset    {ga_cv:.4f} +/- {ga_cv_sd:.4f}")
    print(f"    PROP + LEN   {frozen_cv:.4f} +/- {frozen_cv_sd:.4f}"
          f"   (SE {frozen_cv_se:.4f})")
    paired_a = ga_folds - frozen_folds
    comparable = int(np.count_nonzero(~np.isnan(paired_a)))
    print(f"    gain         {ga_cv - frozen_cv:+.4f}   GA wins "
          f"{int(np.nansum(paired_a > 0))}/{comparable} folds")
    if ga_cv - frozen_cv > frozen_cv_se:
        print("    -> clears one SE of the baseline on this view")
    else:
        print("    -> inside one SE of the baseline: a tie by phase 1's rule,")
        print("       which breaks ties toward the subset with fewer features")
    print()

# ---- 8. Experiment B: the honest test ---------------------------------------
if RUN_B_NESTED:
    print("=" * 78)
    print("B. Nested: the GA searches inside each outer training set only")
    print("=" * 78)
    outer = grouped_folds(all_index, labels, OUTER_SPLITS, RANDOM_SEED)
    ga_outer, frozen_outer, sizes = [], [], []
    for number, (train_index, test_index) in enumerate(outer, start=1):
        if len(np.unique(labels[test_index])) < 2:
            continue
        best, _, _, _ = genetic_search(
            train_index, labels, RANDOM_SEED + number, f"B fold {number}",
        )
        ga_outer.append(roc_auc_score(
            labels[test_index], _fit_predict(train_index, test_index, best, labels)))
        frozen_outer.append(roc_auc_score(
            labels[test_index],
            _fit_predict(train_index, test_index, frozen_mask, labels)))
        sizes.append(int(best.sum()))
        print(f"    fold {number}: GA {ga_outer[-1]:.4f} ({sizes[-1]} columns)   "
              f"PROP + LEN {frozen_outer[-1]:.4f}   "
              f"delta {ga_outer[-1] - frozen_outer[-1]:+.4f}")
    ga_outer = np.array(ga_outer)
    frozen_outer = np.array(frozen_outer)
    delta = ga_outer - frozen_outer
    ga_mean, ga_sd, _ = fold_summary(ga_outer)
    frozen_mean, frozen_sd, frozen_se = fold_summary(frozen_outer)
    delta_mean, delta_sd, _ = fold_summary(delta)
    results["B"] = {
        "ga_mean": ga_mean,
        "ga_std": ga_sd,
        "frozen_mean": frozen_mean,
        "frozen_std": frozen_sd,
        "frozen_se": frozen_se,
        "delta_mean": delta_mean,
        "delta_sd": delta_sd,
        "folds_won": int((delta > 0).sum()),
        "folds": len(delta),
        "mean_features": float(np.mean(sizes)),
    }
    print()
    print("  per outer fold, then mean +/- std as phase 1 reports it:")
    b_table = pd.DataFrame([
        fold_row(f"GA nested ({np.mean(sizes):.0f} col avg)", ga_outer),
        fold_row(f"PROP + LEN ({int(frozen_mask.sum())} col)", frozen_outer),
        fold_row("delta", delta),
    ]).set_index("subset")
    print(b_table.round(4).to_string())
    print()
    print(f"  GA           {ga_mean:.4f} +/- {ga_sd:.4f}  "
          f"({np.mean(sizes):.0f} columns on average)")
    print(f"  PROP + LEN   {frozen_mean:.4f} +/- {frozen_sd:.4f}  "
          f"({int(frozen_mask.sum())} columns, SE {frozen_se:.4f})")
    print(f"  difference   {delta_mean:+.4f} +/- {delta_sd:.4f}   "
          f"GA wins {int((delta > 0).sum())}/{len(delta)} folds")
    print()

# ---- 9. Experiment C: the null control --------------------------------------
if RUN_C_NULL:
    print("=" * 78)
    print("C. Null control: the same GA on shuffled labels")
    print("=" * 78)
    rng = np.random.default_rng(RANDOM_SEED)
    shuffled = labels.copy()
    rng.shuffle(shuffled)
    frozen_null = holdout_score(frozen_mask, all_index, shuffled, RANDOM_SEED)
    best_c, obj_c, _, _ = genetic_search(all_index, shuffled, RANDOM_SEED, "C")
    results["C"] = {
        "frozen_null": frozen_null,
        "ga_null": obj_c,
        "inflation": obj_c - frozen_null,
        "ga_features": int(best_c.sum()),
    }
    print()
    print(f"  labels are shuffled, so no subset is truly better than any other")
    print(f"  PROP + LEN   objective = {frozen_null:.4f}")
    print(f"  GA           objective = {obj_c:.4f}  ({int(best_c.sum())} columns)")
    print(f"  pure selection inflation {obj_c - frozen_null:+.4f}")
    print()

# ---- 10. Verdict ------------------------------------------------------------
print("=" * 78)
print("VERDICT")
print("=" * 78)
rows = []
if "A" in results:
    rows.append(("A  naive, its objective", results["A"]["objective_gain"],
                 "what a naive report would quote"))
    rows.append(("A  naive, 5-fold CV", results["A"]["apparent_gain"],
                 "same subset, scored less optimistically"))
if "C" in results:
    rows.append(("C  null, shuffled labels", results["C"]["inflation"],
                 "what a search gains from noise alone"))
if "B" in results:
    rows.append(("B  nested, held out", results["B"]["delta_mean"],
                 "what the search actually gains"))
print(f"{'experiment':<28}{'GA minus PROP + LEN':>22}   meaning")
print("-" * 78)
for name, value, meaning in rows:
    print(f"{name:<28}{value:>+22.4f}   {meaning}")
print()
if "B" in results and "A" in results:
    if results["B"]["delta_mean"] <= 0:
        print("B is not positive: the subsets the search returns do not generalise")
        print("better than the five frozen columns, however good they looked in A.")
    else:
        print("B is positive. Read it against its spread before claiming a gain:")
        print(f"  delta = {results['B']['delta_mean']:+.4f}, "
              f"sd across folds = {results['B']['delta_sd']:.4f}, "
              f"won {results['B']['folds_won']}/{results['B']['folds']} folds.")
        if results["B"]["delta_mean"] > results["B"]["frozen_se"]:
            print(f"  It clears one SE of the baseline "
                  f"({results['B']['frozen_se']:.4f}), so phase 1's own rule")
            print("  would not call this a tie. Report it, with the spread above.")
        else:
            print(f"  It is inside one SE of the baseline "
                  f"({results['B']['frozen_se']:.4f}), which phase 1 treats as")
            print("  a tie -- and breaks toward the subset with fewer features.")
print()
summary = pd.DataFrame(
    [{"experiment": k, **v} for k, v in results.items()]
)
out = Path("/kaggle/working/metaheuristic_check.csv")
summary.to_csv(out, index=False)
print(summary.to_string(index=False))
print()
print(f"Saved to {out}")
